# 🎙️ VoiceBatch Studio v2.0.4 - [Language Control Fix]
यह वर्जन भाषा बदलने वाली समस्या को जड़ से खत्म करने के लिए बनाया गया है।

In [ ]:
# @title 📥 Step 1: GitHub Root & Engine Setup
import os
REPO_DIR = "VoiceBatch_Storage"
os.makedirs(f"{REPO_DIR}/configs", exist_ok=True)

print("⏳ लाइब्रेरी इंस्टॉल हो रही हैं...")
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts

# 1000% Language Control File बनाना
lang_fix = "def check_lang(t, l): return t.strip()"
with open(f'{REPO_DIR}/configs/lang_control.py', 'w') as f: f.write(lang_fix)

print("✅ इंजन और कंट्रोल फाइल तैयार है!")

In [ ]:
# @title 🚀 Step 2: app.py (Strict Language Enforcement)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading XTTS Engine on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def voice_pro_engine(text, audio_sample, speed, pitch, lang):
    if not audio_sample: return None
    
    temp_out = 'VoiceBatch_Storage/raw_output.wav'
    
    # भाषा को यहाँ कड़ाई से बांधा गया है
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=temp_out
    )
    
    # ऑडियो को साफ़ और स्मूथ बनाना
    y, sr = librosa.load(temp_out)
    y, _ = librosa.effects.trim(y, top_db=25)
    
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_path = 'VoiceBatch_Storage/v_batch_pro.wav'
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.4')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='यहाँ स्क्रिप्ट लिखें (हिंदी)', lines=5)
            smp = gr.Audio(label='वॉइस सैंपल अपलोड करें', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr', 'bn', 'gu'], label='Select Language (Strict)', value='hi')
            with gr.Row():
                spd = gr.Slider(0.8, 1.2, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-3, 3, 0, step=1, label="Pitch")
            btn = gr.Button('Realistic Voice Generate 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='फाइनल आउटपुट')

    btn.click(voice_pro_engine, [txt, smp, spd, ptc, lng], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ app.py अपडेट हो गया। अब कोई दूसरी भाषा बीच में नहीं आएगी।")
!python app.py